# 環境変数

In [ ]:
import os
os.environ["ERG_DATA_DIR"] = "/mnt/j/observation_data/"

# 3dfluxデータを時間軸に焼き直す

In [ ]:
import pyspedas as psp
import pytplot as pt
import numpy as np
import xarray as xr

pt.del_data('*')

time_range_full = ['2017-11-15/16:10:00', '2017-11-15/16:30:00']
psp.erg.lepi(time_range_full, datatype='3dflux', get_support_data=True, no_update=True)

# 解析対象の狭い時間窓
time_range = ['2017-11-15/16:15:00', '2017-11-15/16:30:00']

# flux3d本体 (dims = time, v1(energy), v2(channel), v3(phase))  # [#/cm2/sr/sec/keV]
flux3d = pt.data_quants['erg_lepi_l2_3dflux_FPDU'].sel(
    time=slice(*time_range)
)

# 各軸の座標を取り出しておく
time_ax     = flux3d.time.values        # shape = (T,)
energy_ax   = flux3d.v1.values          # (E,)
channel_ax  = flux3d.v2.values          # (C,)
spin_ax     = flux3d.v3.values          # (S,)

time_num, energy_num, channel_num, spin_num = len(time_ax), len(energy_ax), len(channel_ax), len(spin_ax)   # T, E, C, S

# 1 spin time = 8 sec
# 1 spin phase time = 0.5 sec
# 1 energy time step = 15625 μsec
spin_offset_ns      = np.arange(spin_num, dtype='timedelta64[ns]') * 500_000_000        # 0.5 sec = 500,000,000 nsec
energy_offset_ns    = (np.arange(energy_num, dtype='int64') * 15_625_000 + 7_812_500).astype('timedelta64[ns]')

offset_ns           = energy_offset_ns[:, None] + spin_offset_ns    # (E, 1) + (S) -> (E, S)

flux_E_TS_C = flux3d.transpose('v1_dim', 'time', 'v3_dim', 'v2_dim').values.reshape(energy_num, time_num*spin_num, channel_num)

# xarray.DataArrayをエネルギーごとに生成
flux_data_arrays = {}
for energy_i in range(energy_num):
    time_flat = (time_ax[:, None] + offset_ns[energy_i][None, :]).reshape(-1) # ((T, 1) + (1, S) -> (T, S)).reshape(-1) -> (TxS,)

    flux_data_arrays[energy_i] = xr.DataArray(
        flux_E_TS_C[energy_i],
        dims=['time', 'channel'],
        coords={
            'time':         time_flat,
            'channel':      channel_ax,
            'energy_keV':   energy_ax[energy_i]
        },
        attrs=flux3d.attrs,
        name=f'flux_energy_{energy_i}'
    )

fidu_angle_dict     = pt.data_quants['erg_lepi_l2_3dflux_FIDU_Angle_sga']
fidu_angle  = fidu_angle_dict['data'].astype(float)     # shape (2, 3, 16)
AZ_deg_mid = fidu_angle[0, 1, :channel_num]      # shape (C,)
# AZ_deg_mid =  [ 78.75  56.25  33.75  11.25 -11.25 -33.75 -56.25 -78.75]

theta_sga = AZ_deg_mid                  # (C,)
varphi_sga = -90. * np.ones(spin_num)   # (S,)

# (TxS, C)の2次元配列を生成
theta_sga_time = np.tile(theta_sga, (time_num*spin_num, 1))   # (TxS, C)
varphi_sga_times = np.tile(varphi_sga, (time_num, 1)).reshape(-1, 1)   # (TxS, 1)
varphi_sga_time = np.tile(varphi_sga_times, (1, channel_num))   # (TxS, C)

angle_sga_time = np.stack((theta_sga_time, varphi_sga_time), axis=2)   # (TxS, C, 2)

# theta_sga, varphi_sga -> Vx_sga, Vy_sga, Vz_sga
vector_sga_time = np.zeros((time_num*spin_num, channel_num, 3))   # (TxS, C, 3)
vector_sga_time[:, :, 0] = np.cos(np.radians(angle_sga_time[:, :, 0])) * np.cos(np.radians(angle_sga_time[:, :, 1]))
vector_sga_time[:, :, 1] = np.cos(np.radians(angle_sga_time[:, :, 0])) * np.sin(np.radians(angle_sga_time[:, :, 1]))
vector_sga_time[:, :, 2] = np.sin(np.radians(angle_sga_time[:, :, 0]))

v_unit_vector_sga_energy_channle_list = {}
for energy_i in range(energy_num):
    time_flat = (time_ax[:, None] + offset_ns[energy_i][None, :]).reshape(-1)

    for channel_i in range(channel_num):
        v_unit_vector_sga_energy_channle_list[energy_i, channel_i] = xr.DataArray(
            vector_sga_time[:, channel_i, :],
            dims=['time', 'xyz'],
            coords={
                'time': time_flat,
                'xyz': ['x', 'y', 'z'],
                'channel': channel_ax[channel_i],
                'energy_keV':   energy_ax[energy_i]
            },
            name=f'v_unit_vector_sga_{energy_i}_{channel_i}'
        )
        print(f'v_unit_vector_sga_energy_channle_list[{energy_i, channel_i}] = ', v_unit_vector_sga_energy_channle_list[energy_i, channel_i])

# SGA座標系→DSI座標系に変換

In [ ]:
import pyspedas as psp
import pytplot as pt
import xarray as xr

v_unit_vector_dsi_energy_channle_list = {}
for energy_i in range(energy_num):
    for channel_i in range(channel_num):
        pt.store_data(f'vector_sga_{energy_i}_{channel_i}', data={'x': time_flat, 'y': v_unit_vector_sga_energy_channle_list[energy_i, channel_i].values})
        print(pt.data_quants[f'vector_sga_{energy_i}_{channel_i}'])
        # dsi座標系に変換
        psp.projects.erg.erg_cotrans(f'vector_sga_{energy_i}_{channel_i}', f'vector_dsi_{energy_i}_{channel_i}', in_coord='sga', out_coord='dsi')
        v_unit_vector_dsi_energy_channle_list[energy_i, channel_i] = xr.DataArray(
            pt.data_quants[f'vector_dsi_{energy_i}_{channel_i}'].values,
            dims=['time', 'xyz'],
            coords={
                'time': time_flat,
                'xyz': ['x', 'y', 'z'],
                'channel': channel_ax[channel_i],
                'energy_keV':   energy_ax[energy_i]
            },
            name=f'v_unit_vector_dsi_{energy_i}_{channel_i}'
        )
        print(f'v_unit_vector_dsi_energy_channel_list[{energy_i}, {channel_i}] = ', v_unit_vector_dsi_energy_channle_list[energy_i, channel_i])

# 背景磁場ベクトルとその単位ベクトルの決定

In [ ]:
import pyspedas as psp
import pytplot as pt
import xarray as xr
import numpy as np

psp.erg.mgf(trange=time_range_full, level='l2', datatype='8sec', coord='dsi', no_update=True)
B_8sec = pt.data_quants['erg_mgf_l2_mag_8sec_dsi']

In [ ]:
background_time_sec = 100 #[sec]

In [ ]:
B0_unit_vector_dsi_energy_channle_list = {}
for energy_i in range(energy_num):
    for channel_i in range(channel_num):
        B_8sec_interp       = B_8sec.interp(time=v_unit_vector_dsi_energy_channle_list[energy_i, channel_i].time, method='linear')
        dt_B_8sec_interp    = (B_8sec_interp.time[1] - B_8sec_interp.time[0]) / np.timedelta64(1, 's')
        B_background        = B_8sec_interp.rolling(time=int(background_time_sec/dt_B_8sec_interp), center=True).mean('time')

        unit_B_background   = B_background / np.sqrt(B_background[:, 0]**2E0 + B_background[:, 1]**2E0 + B_background[:, 2]**2E0)

        B0_unit_vector_dsi_energy_channle_list[energy_i, channel_i] = xr.DataArray(
            unit_B_background.data,
            dims=['time', 'xyz'],
            coords={
                'time': unit_B_background.time,
                'xyz': ['x', 'y', 'z'],
                'channel': channel_ax[channel_i],
                'energy_keV':   energy_ax[energy_i]
            },
            name=f'B0_unit_vector_dsi_{energy_i}_{channel_i}'
        )
        print(f'B0_unit_vector_dsi_energy_channle_list[{energy_i}, {channel_i}] = ', B0_unit_vector_dsi_energy_channle_list[energy_i, channel_i])

# 対象とする垂直磁場成分を取得

In [ ]:
psp.erg.mgf(trange=time_range_full, level='l2', datatype='64hz', coord='dsi', no_update=True)

B_64Hz  = pt.data_quants['erg_mgf_l2_mag_64hz_dsi']
t_B_64Hz    = B_64Hz.time
dt_B_64Hz = (t_B_64Hz[1] - t_B_64Hz[0]) / np.timedelta64(1, 's')
B_8sec_interp   = B_64Hz.interp(time=t_B_64Hz, method='linear')
B_background    = B_8sec_interp.rolling(time=int(background_time_sec/dt_B_64Hz), center=True).mean('time')

B_background_unit   = B_background / np.sqrt(B_background[:, 0]**2E0 + B_background[:, 1]**2E0 + B_background[:, 2]**2E0)

In [ ]:
B_background_unit = B_background_unit.rename({"v_dim": "xyz"})

In [ ]:
B_64Hz_perp = B_64Hz - (B_64Hz * B_background_unit).sum(dim='xyz')
print(B_64Hz_perp[90000:90050, :])
print(B_64Hz[90000:90050, :])